In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd
import os
from scipy.stats import gaussian_kde
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors
from matplotlib import colors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
lat = np.arange(-40, 41, 1)
lon = np.arange(0, 361, 1)

TC_path1d = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/genesis-new-1d/'
TC_path2d = r'F:\study\data\noaa-20c\NOAA\NOAA_20C_TXT_30_all_bing\genesis-new-2d\\'
TCT_path = r'../../../study/data/noaa-20c/NOAA/NOAA_20C_TXT_30_all_bing/'
soda_path = r'F:\study\data\soda\\'
path_noaa = r'F:\study\data\noaa-20c\\'

In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2014-01')]
ep = ep[ep < np.datetime64('2014-01')]
mix = mix[mix < np.datetime64('2014-01')]
suc = suc[suc < np.datetime64('2014-01')]
events = [mix, cp, ep, suc]

# tcg_grid = xr.open_dataset('./data-20c/NOAA-20C_TC_grid.nc').tcg_grid.loc['1900':'2015']
# tct_grid = xr.open_dataset('./data-20c/NOAA-20C_TCT_grid.nc').tct_grid.loc['1900':'2015', -40:40]
ssta = xr.open_dataset('./data/ssta.nc').sst.loc['1900':'2015', -40:40]

rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()

# tcg_grid -= tcg_grid.groupby('time.month').map(rolling)
# tct_grid -= tct_grid.groupby('time.month').map(rolling)
ssta -= ssta.groupby('time.month').map(rolling)

ssta_sum = ssta.sel(time=ssta.time.dt.month.isin([6, 7, 8, 9, 10, 11])).coarsen(time=6).mean()
ssta_win = ssta[11:-1].sel(time=ssta[11:-1].time.dt.month.isin([1, 2, 3, 4, 5, 12])).coarsen(time=6).mean()

In [ ]:
def composite_ttest(ssta_sum, event):
    mask = ssta_sum.time.dt.year.isin(event.astype('datetime64[Y]').astype(str).astype(int))
    ssta_mar = ssta_sum.where(mask, drop=True)
    ssta_nor = ssta_sum.where(~mask, drop=True)

    n1 = ssta_mar.shape[0]
    n2 = ssta_nor.shape[0]

    m1 = ssta_mar.mean(dim='time')
    m2 = ssta_nor.mean(dim='time')
    v1 = ssta_mar.var(dim='time', ddof=1)
    v2 = ssta_nor.var(dim='time', ddof=1)

    t_stat = (m1 - m2)/np.sqrt(v1/n1 + v2/n2)
    df = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
    p_value = stats.t.sf(np.abs(t_stat), df)
    p_value = xr.DataArray(p_value, coords=t_stat.coords)
    return m1, p_value

In [ ]:
ny, nx = len(lat), len(lon)
ds_nino = xr.Dataset(
    {
        'ssta_sum':(['type', 'stage', 'lat', 'lon'], np.empty((4, 2, ny, nx))),
        'ssta_win':(['type', 'stage', 'lat', 'lon'], np.empty((4, 2, ny, nx))),
        'p_sum':(['type', 'stage', 'lat', 'lon'], np.empty((4, 2, ny, nx))),
        'p_win':(['type', 'stage', 'lat', 'lon'], np.empty((4, 2, ny, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'CP', 'EP', 'SUC']),
        'stage':(['stage'], ['current', 'next']),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)

for i in range(4):
    event = events[i]
    ds_nino.ssta_sum[i, 0, :, :], ds_nino.p_sum[i, 0, :, :] = composite_ttest(ssta_sum, event)
    ds_nino.ssta_sum[i, 1, :, :], ds_nino.p_sum[i, 1, :, :] = composite_ttest(ssta_sum, event+np.timedelta64(1, 'Y'))
    ds_nino.ssta_win[i, 0, :, :], ds_nino.p_win[i, 0, :, :] = composite_ttest(ssta_win, event+np.timedelta64(1, 'Y'))
    ds_nino.ssta_win[i, 1, :, :], ds_nino.p_win[i, 1, :, :] = composite_ttest(ssta_win, event+np.timedelta64(2, 'Y'))
    
ds_nino.to_netcdf('./significant/composite_ssta_season.nc')

In [ ]:
p_value.where(p_value < 0.1).plot.contourf(levels=21)